# Preparación del dataset para fine-tuning QLoRA

Este notebook convierte el dataset de clasificación al formato instrucción–respuesta
necesario para el fine-tuning con QLoRA (notebook 12).

Pasos:
1. Cargar `data/dataset_riesgo.csv` (500 muestras)
2. Formatear cada muestra como par instrucción–respuesta
3. Split train (80%) / test (20%) estratificado
4. Guardar en `data/finetune/train.jsonl` y `data/finetune/test.jsonl`
5. Registro en MLflow

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Carga del dataset

In [3]:
import pandas as pd

df = pd.read_csv("data/eu_ai_act_flagged_es.csv")

print(f"Total de muestras: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print()
print("Distribución por clase:")
print(df["etiqueta"].value_counts())

Total de muestras: 300
Columnas: ['descripcion', 'violation', 'category', 'severity', 'articles', 'explanation', 'context', 'split', 'ambiguity', 'etiqueta']

Distribución por clase:
etiqueta
inaceptable        131
riesgo_minimo      103
alto_riesgo         46
riesgo_limitado     20
Name: count, dtype: int64


## 2. Formato instrucción–respuesta

Cada muestra se convierte al formato:
```
### Instrucción:
Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial de la UE
en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo.

### Descripción:
{descripcion}

### Clasificación:
{etiqueta}
```

In [4]:
INSTRUCCION = (
    "Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial "
    "de la UE en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo."
)

def formatear_muestra(row):
    return (
        f"### Instrucción:\n{INSTRUCCION}\n\n"
        f"### Descripción:\n{row['descripcion']}\n\n"
        f"### Clasificación:\n{row['etiqueta']}"
    )

df["text"] = df.apply(formatear_muestra, axis=1)

print("Ejemplo de muestra formateada:")
print("-" * 60)
print(df["text"].iloc[0])

Ejemplo de muestra formateada:
------------------------------------------------------------
### Instrucción:
Clasifica el siguiente sistema de IA según el Reglamento de Inteligencia Artificial de la UE en una de estas categorías: inaceptable, alto_riesgo, riesgo_limitado, riesgo_minimo.

### Descripción:
Una ciudad inteligente implementa un sistema de inteligencia artificial para la optimización del semáforo que incluye un panel público que explica el propósito del sistema, las fuentes de datos y la lógica de toma de decisiones, con actualizaciones en tiempo real sobre cómo los ajustes de la inteligencia artificial impactan el flujo de tráfico.

### Clasificación:
riesgo_minimo


## 3. Split train / test estratificado

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["etiqueta"],
)

print(f"Train: {len(train_df)} muestras")
print(f"Test:  {len(test_df)} muestras")
print()
print("Distribución train:")
print(train_df["etiqueta"].value_counts())
print()
print("Distribución test:")
print(test_df["etiqueta"].value_counts())

Train: 240 muestras
Test:  60 muestras

Distribución train:
etiqueta
inaceptable        105
riesgo_minimo       82
alto_riesgo         37
riesgo_limitado     16
Name: count, dtype: int64

Distribución test:
etiqueta
inaceptable        26
riesgo_minimo      21
alto_riesgo         9
riesgo_limitado     4
Name: count, dtype: int64


## 4. Guardar en formato JSONL

In [6]:
import json
import os

output_dir = "data/finetune"
os.makedirs(output_dir, exist_ok=True)

def guardar_jsonl(df, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            record = {
                "text":     row["text"],
                "etiqueta": row["etiqueta"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

path_train = os.path.join(output_dir, "train.jsonl")
path_test  = os.path.join(output_dir, "test.jsonl")

guardar_jsonl(train_df, path_train)
guardar_jsonl(test_df,  path_test)

print(f"✓ Train guardado en: {path_train} ({len(train_df)} muestras)")
print(f"✓ Test  guardado en: {path_test}  ({len(test_df)} muestras)")

✓ Train guardado en: data/finetune\train.jsonl (240 muestras)
✓ Test  guardado en: data/finetune\test.jsonl  (60 muestras)


## 5. Verificación — longitud de secuencias

Comprobamos que las muestras caben en el contexto del modelo base (`max_length=512`).

In [7]:
# Estimación aproximada de tokens (1 token ≈ 4 caracteres en español)
df["tokens_aprox"] = df["text"].str.len() / 4

print("Estadísticas de longitud aproximada en tokens:")
print(df["tokens_aprox"].describe().round(1))
print()
superan_512 = (df["tokens_aprox"] > 512).sum()
print(f"Muestras que superan 512 tokens (aprox.): {superan_512} de {len(df)}")
if superan_512 > 0:
    print("⚠ Estas muestras serán truncadas durante la tokenización.")

Estadísticas de longitud aproximada en tokens:
count    300.0
mean     140.0
std       16.1
min       72.2
25%      129.8
50%      140.1
75%      147.8
max      201.8
Name: tokens_aprox, dtype: float64

Muestras que superan 512 tokens (aprox.): 0 de 300


## 6. Registro en MLflow

In [8]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
import mlflow
from functions import configure_mlflow, MLFLOW_EXPERIMENT

try:
    configure_mlflow()
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="preparacion_dataset_finetune"):
        mlflow.log_params({
            "n_muestras":    len(df),
            "formato":       "instruccion",
            "train_split":   0.8,
            "n_clases":      df["etiqueta"].nunique(),
            "max_length":    512,
        })

        for clase, n in df["etiqueta"].value_counts().items():
            mlflow.log_metric(f"n_{clase}", n)

        mlflow.log_metric("n_train", len(train_df))
        mlflow.log_metric("n_test",  len(test_df))
        mlflow.log_metric("tokens_aprox_media", df["tokens_aprox"].mean())

        mlflow.log_artifact(path_train)
        mlflow.log_artifact(path_test)

        print("✓ Dataset de fine-tuning registrado en MLflow")
        print(f"  Run ID: {mlflow.active_run().info.run_id}")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://34.244.146.100
⚠ MLflow no disponible: API request to https://34.244.146.100/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPSConnectionPool(host='34.244.146.100', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_ia_artificial (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)')))
